# Training the turnover model

This notebook is a visual, exploratory walkthrough of the **exact** pipeline `scripts/train_turnover_model.py` and the webapp's `/model/train` endpoint run in production (`companysim.ml.gate.run_training_gate`). It does not reimplement any training logic — every step below calls the same tested functions from `companysim.ml.*`, it just adds inspection and charts around them.

Companion notebook: **`02_evaluate_turnover_model.ipynb`** evaluates a trained bundle against the project's fixed holdout benchmark with ROC/PR curves, calibration, and feature importance.

**Pipeline**: generate a day-0 synthetic population &rarr; forward-simulate it several times to get *real* stochastic exit-event labels &rarr; build leakage-free observable features &rarr; train a `GradientBoostingClassifier`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / 'pyproject.toml').exists():
    os.chdir('..')
print(f'repo root: {Path.cwd()}')

repo root: D:\companysim


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from companysim.data.datasets import DatasetConfig
from companysim.ml.diagnostics import _aggregate_feature_importances
from companysim.ml.registry import save_bundle
from companysim.ml.train import train_turnover_model
from companysim.ml.turnover_features import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    build_feature_frame,
)
from companysim.ml.turnover_labels import build_turnover_cohort

pd.set_option('display.max_columns', None)

## 1. Build a training cohort

`build_turnover_cohort` generates a population once, then forward-simulates it multiple times (`replicates`) so the *same* day-0 features get several independent stochastic label draws — this is what keeps the model from learning a deterministic feature&rarr;label mapping. See `ml/turnover_labels.py` for the full rationale.

Sized for a notebook run (a couple of minutes); production training (`scripts/train_turnover_model.py`'s defaults) uses `headcount=2000, replicates=4`.

In [3]:
TRAIN_HEADCOUNT = 1500
TRAIN_REPLICATES = 3
TRAIN_HORIZON = 12
TRAIN_SEED = 2024

train_cfg = DatasetConfig(name='notebook_train', headcount=TRAIN_HEADCOUNT, seed=TRAIN_SEED)
cohort = build_turnover_cohort(train_cfg, horizon_ticks=TRAIN_HORIZON, replicates=TRAIN_REPLICATES)

print(f'{len(cohort.labels):,} labeled rows ({TRAIN_HEADCOUNT:,} employees x {TRAIN_REPLICATES} replicates)')
cohort.labels.head()

4,500 labeled rows (1,500 employees x 3 replicates)


,employee_id,replicate,quit_within_horizon,tick_of_exit
0,emp_000000,0,False,NaN
1,emp_000001,0,False,NaN
2,emp_000002,0,False,NaN
3,emp_000003,0,False,NaN
4,emp_000004,0,False,NaN


## 2. Inspect the label distribution


In [4]:
quit_rate = cohort.labels['quit_within_horizon'].mean()
print(f'Quit-within-horizon rate: {quit_rate:.2%}')

counts = cohort.labels['quit_within_horizon'].value_counts().rename({False: 'Stayed', True: 'Quit'})
fig = px.bar(
    x=counts.index, y=counts.values, color=counts.index,
    labels={'x': '', 'y': 'Employees x replicates'},
    title=f'Label balance ({quit_rate:.1%} positive rate)',
    color_discrete_map={'Stayed': '#4f46e5', 'Quit': '#dc2626'},
)
fig.update_layout(showlegend=False, template='plotly_white')
fig.show()

Quit-within-horizon rate: 6.62%


In [5]:
quit_by_tick = cohort.labels.dropna(subset=['tick_of_exit'])['tick_of_exit']
fig = px.histogram(
    quit_by_tick, nbins=TRAIN_HORIZON,
    labels={'value': 'Week of exit'},
    title='When do the (labeled) quits actually happen within the horizon?',
)
fig.update_layout(showlegend=False, template='plotly_white')
fig.show()

## 3. Build leakage-free features

`build_feature_frame` only uses things a real HRIS/pulse-survey platform could observe *before* the outcome window — see the leakage boundary documented in `ml/turnover_features.py`. `assert_no_leakage` (called internally by `train_turnover_model`) is a standing guard against regression.

In [6]:
feats = build_feature_frame(cohort.tables)
merged = cohort.labels.merge(feats, on='employee_id')
merged[list(FEATURE_COLUMNS)].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
level,4500,10,IC3,939,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department_id,4500,9,dept_00,1392,NaN,NaN,NaN,NaN,NaN,NaN,NaN
role,4500,54,Frontend Engineer,174,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure_months,4500.0,NaN,NaN,NaN,34.68,24.293279,0.0,18.0,30.0,45.0,231.0
base_salary,4500.0,NaN,NaN,NaN,140348.517247,68231.750247,30000.0,93340.715,130683.42,172090.2725,669677.83
team_size,4500.0,NaN,NaN,NaN,6.925333,1.824417,1.0,6.0,7.0,8.0,11.0
is_manager,4500.0,NaN,NaN,NaN,0.277333,0.447732,0.0,0.0,0.0,1.0,1.0
promotions_count,4500.0,NaN,NaN,NaN,1.518667,1.081624,0.0,1.0,1.0,2.0,6.0
mood_pulse_mean,4500.0,NaN,NaN,NaN,0.528005,0.113083,0.154425,0.452206,0.528438,0.607625,0.8584
stress_level_pulse_mean,4500.0,NaN,NaN,NaN,0.362247,0.135962,0.0,0.266838,0.352625,0.45545,0.902925


## 4. Train


In [7]:
bundle, report = train_turnover_model(merged, seed=TRAIN_SEED)
pd.DataFrame([report.as_dict()]).T.rename(columns={0: 'value'})

,value
auc,0.619854
precision_at_10,0.089286
precision_at_20,0.111111
base_rate,0.066222
n_train,3375.000000
n_test,1125.000000


## 5. Feature importance

Reuses the project's own aggregation helper (`ml.diagnostics._aggregate_feature_importances`) that sums a fitted pipeline's one-hot-expanded importances back to their parent (pre-encoding) feature names — the same weighting the root-cause diagnosis pipeline uses.

In [8]:
importances = _aggregate_feature_importances(bundle.classifier, CATEGORICAL_FEATURES)
imp_df = (
    pd.Series(importances, name='importance')
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={'index': 'feature'})
)
fig = px.bar(
    imp_df, x='importance', y='feature', orientation='h',
    title='Feature importance (train split)', template='plotly_white',
)
fig.update_traces(marker_color='#4f46e5')
fig.show()

## 6. Save the trained candidate

Saved under a **distinct filename**, deliberately *not* `models/turnover_production.joblib` — this notebook trains and inspects a candidate, it does not deploy it. Promotion to production only ever happens through `ml.gate.run_training_gate` (CLI: `scripts/train_turnover_model.py`, or the webapp's Train Model page), which evaluates the candidate against the fixed holdout first and only promotes if it doesn't regress AUC — see `02_evaluate_turnover_model.ipynb` for exactly that evaluation.

In [9]:
candidate_path = Path('models/turnover_candidate_from_notebook.joblib')
save_bundle(bundle, candidate_path)
print(f'Saved candidate -> {candidate_path.resolve()}')

Saved candidate -> D:\companysim\models\turnover_candidate_from_notebook.joblib
